# Stage 6 — FRANK: The ONE Model

**Goal:** a single architecture that gets the best ARI on all six datasets.

**What FRANK combines:**
1. **SAGE encoders** per modality (Stage 4 champion)
2. **Per-spot MoE fusion** with threshold pruning γ=0.3 (GROVER) — the gate that doesn't collapse
3. **Shared-private 16+8+8** (Stage 5 proven)
4. **Homogeneity-gated spatial** (OmicSync-style) — auto spatial strength per dataset, no switches
5. **Masked reconstruction 15%** (scMRDR) — not identity mapping
6. **Isometric regularization** (scMRDR) — anti-collapse with teeth
7. **Spherical k-means** default head (Exp0: mouse needs it)

**Firewall:** no labels in training, preprocessing, gating, or selection. Labels → posthoc.json only.
**Selection:** RAUS v3b (Borda{stability, recbal, R²_rna}).


In [ ]:
import os, sys, json, subprocess, random, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for pkg in ["scanpy", "anndata", "scikit-learn", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import scanpy as sc
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.neighbors import kneighbors_graph
from scipy import sparse
import pandas as pd

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

print("imports ok")


In [ ]:
# ---------- GitHub push (same pattern as stage5) ----------
REPO = "/kaggle/working/my_research"
REPO_URL = "https://github.com/rifatahsanpul0k/my_research.git"
def sh(*args):
    subprocess.run(list(args), check=True, cwd=REPO)
os.makedirs(REPO, exist_ok=True)
OUT = os.path.join(REPO, "runs", "s6-frank")
os.makedirs(OUT, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    import base64, shutil
    _sec = UserSecretsClient()
    _b64 = base64.b64encode(f"x-access-token:{_sec.get_secret('GITHUB_TOKEN')}".encode()).decode()
    _GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {_b64}"]
    if not os.path.isdir(os.path.join(REPO, ".git")):
        if os.listdir(REPO):
            bak = REPO + "_stalebak"
            if os.path.exists(bak):
                shutil.rmtree(bak)
            shutil.move(REPO, bak)
            os.makedirs(REPO, exist_ok=True)
        subprocess.run(_GIT + ["clone", REPO_URL, REPO], check=True)
        sh("git", "config", "user.name", "kaggle-runner")
        sh("git", "config", "user.email", "kaggle-runner@local")
    else:
        sh(*_GIT, "pull", "--rebase", "origin", "main")
    GIT_OK = True
    print("GitHub sync ready")
except Exception as e:
    GIT_OK = False
    _safe = str(e).replace(_b64, "***") if "_b64" in dir() else str(e)
    print("GitHub sync unavailable (local save only):", _safe)

def _scrub(s):
    try:
        return str(s).replace(_b64, "***")
    except NameError:
        return str(s)

def git_push(paths, msg):
    if not GIT_OK:
        return
    try:
        sh("git", "add", *paths)
        r = subprocess.run(["git", "status", "--porcelain"], capture_output=True,
                           text=True, cwd=REPO)
        if r.stdout.strip():
            sh("git", "commit", "-m", msg)
        sh(*_GIT, "pull", "--rebase", "origin", "main")
        sh(*_GIT, "push", "origin", "main")
        print("pushed:", msg)
    except Exception as e:
        print("push failed (continuing):", _scrub(e))


In [ ]:
# ---------- data (same as stage5) ----------
MARKER = "10x_human_lymph_node_A1"
def find_data_root():
    for base in ["/kaggle/input/spatial-multiomics-6datasets-private",
                 "/kaggle/working/spatial-multiomics-6datasets-private",
                 "/kaggle/working"]:
        if os.path.isdir(os.path.join(base, MARKER)):
            return base
    return None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    kd = os.path.expanduser("~/.kaggle"); os.makedirs(kd, exist_ok=True)
    json.dump({"username": _s.get_secret("KAGGLE_USERNAME"), "key": _s.get_secret("KAGGLE_KEY")},
              open(os.path.join(kd, "kaggle.json"), "w"))
    os.chmod(os.path.join(kd, "kaggle.json"), 0o600)
    subprocess.run(["kaggle", "datasets", "download", "-d", "pulokpulok/spatial-multiomics-6datasets-private",
                    "-p", "/kaggle/working", "--unzip"], check=True)
    DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    raise FileNotFoundError("dataset folders not found")
print("DATA_ROOT:", DATA_ROOT)

DATASETS = {
    "A1":  dict(folder="10x_human_lymph_node_A1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "D1":  dict(folder="10x_human_lymph_node_D1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "E11": dict(folder="Mouse_Brain_E11_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E13": dict(folder="Mouse_Brain_E13_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E15": dict(folder="Mouse_Brain_E15_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E18": dict(folder="Mouse_Brain_E18_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
}
DECLARED_K = {"A1": 10, "D1": 11, "E11": 8, "E13": 12, "E15": 12, "E18": 14}
SEEDS = [42, 1234, 2024]


In [ ]:
# ---------- preprocessing (identical to stage5) ----------
def to_dense_small(X, max_elems=5e7):
    if hasattr(X, "toarray"):
        return X.toarray() if X.size <= max_elems else X
    return np.asarray(X)

def clr(X):
    X = np.asarray(X, dtype=np.float64)
    gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
    return np.log1p(X / gm)

def load_dataset(ds_id):
    spec = DATASETS[ds_id]
    root = os.path.join(DATA_ROOT, spec["folder"])
    rna = sc.read_h5ad(os.path.join(root, "adata_RNA.h5ad"))
    rna.var_names_make_unique()
    coords = np.asarray(rna.obsm["spatial"], dtype=np.float64)
    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, n_top_genes=3000, flavor="seurat", inplace=True)
    rna_hvg = rna[:, rna.var["highly_variable"]].copy()
    sc.pp.scale(rna_hvg, max_value=10)
    Xr = PCA(n_components=64, random_state=42).fit_transform(to_dense_small(rna_hvg.X))
    Xr = StandardScaler().fit_transform(Xr)
    if spec["aux"] == "adt":
        aux = sc.read_h5ad(os.path.join(root, "adata_ADT.h5ad"))
        Xa = StandardScaler().fit_transform(clr(to_dense_small(aux.X)))
    else:
        atac = sc.read_h5ad(os.path.join(root, "adata_ATAC.h5ad"))
        tfidf = to_dense_small(atac.X)
        lsi = TruncatedSVD(n_components=50, random_state=42).fit_transform(tfidf)
        depth = np.log1p(np.asarray(atac.X.sum(axis=1)).ravel())
        keep = [i for i in range(50) if abs(np.corrcoef(lsi[:, i], depth)[0, 1]) <= 0.75]
        Xa = StandardScaler().fit_transform(lsi[:, keep])
    # spatial kNN graph
    A = kneighbors_graph(coords, n_neighbors=15, include_self=False, mode="connectivity")
    A = A.maximum(A.T)
    ei = np.array(A.nonzero())
    g = {"edge_index": torch.tensor(ei, dtype=torch.long, device=DEVICE)}
    # homogeneity: mean neighbor cosine similarity in RNA space (OmicSync-style)
    Xn = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-8)
    sims = []
    for i in range(min(2000, len(Xn))):
        nbrs = ei[1, ei[0] == i]
        if len(nbrs):
            sims.append((Xn[i] * Xn[nbrs]).sum(axis=1).mean())
    homog = float(np.mean(sims)) if sims else 0.0
    # gate lam_spa: homogeneous -> stronger spatial; heterogeneous -> weaker
    lam_spa = 0.20 if homog > 0.6 else (0.10 if homog > 0.3 else 0.05)
    print(f"{ds_id}: homog={homog:.3f} -> lam_spa={lam_spa}")
    return dict(Xr=torch.tensor(Xr, dtype=torch.float32, device=DEVICE),
                Xa=torch.tensor(Xa, dtype=torch.float32, device=DEVICE),
                g=g, lam_spa=lam_spa, n=len(Xr))


In [ ]:
# ---------- S6-FRANK model ----------
class SAGELayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out)
        self.lin_n = nn.Linear(d_in, d_out)
    def forward(self, x, g):
        ei = g["edge_index"]
        agg = torch.zeros_like(x)
        agg.index_add_(0, ei[1], x[ei[0]])
        deg = torch.bincount(ei[1], minlength=x.shape[0]).clamp_min(1).unsqueeze(1)
        return F.relu(self.lin(x) + self.lin_n(agg / deg))

def laplacian_loss(z, ei):
    d = (z[ei[0]] - z[ei[1]]).pow(2).sum(dim=1).mean()
    return d

class S6FrankModel(nn.Module):
    # FRANK: SAGE encoders + MoE threshold fusion (GROVER) + shared-private 16+8+8
    #      + masked recon 15% + isometric reg (scMRDR) + homogeneity-gated spatial
    def __init__(self, d_rna, d_aux, d_recon_rna):
        super().__init__()
        self.sage_r = SAGELayer(d_rna, 32)
        self.sage_a = SAGELayer(d_aux, 32)
        # MoE gate: per-spot softmax over [rna, aux], threshold 0.3, renormalize
        self.gate = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))
        self.exp_r = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 32))
        self.exp_a = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 32))
        # shared-private on fused
        self.lin_s = nn.Linear(32, 16)
        self.lin_pr = nn.Linear(32, 8)
        self.lin_pa = nn.Linear(32, 8)
        self.dec_r = nn.Linear(32, d_recon_rna)
        self.dec_a = nn.Linear(32, d_aux)

    def forward(self, xr, xa, g):
        zr = self.sage_r(xr, g)
        za = self.sage_a(xa, g)
        gw = F.softmax(self.gate(torch.cat([zr, za], dim=1)), dim=1)
        gw = gw * (gw >= 0.3).float()          # threshold pruning (GROVER)
        gw = gw / gw.sum(dim=1, keepdim=True).clamp_min(1e-8)
        zf = gw[:, 0:1] * self.exp_r(zr) + gw[:, 1:2] * self.exp_a(za)
        zs = self.lin_s(zf)
        zpr = self.lin_pr(zr)   # private from modality-specific
        zpa = self.lin_pa(za)
        z = torch.cat([zs, zpr, zpa], dim=1)
        return z, zs, zpr, zpa, gw

    def loss(self, out, xr, xa, tr, ta, g, lam_spa):
        z, zs, zpr, zpa, gw = out
        # masked recon 15% (not identity mapping)
        mr = (torch.rand_like(tr) > 0.15).float()
        ma = (torch.rand_like(ta) > 0.15).float()
        L = F.mse_loss(self.dec_r(z) * mr, tr * mr) + F.mse_loss(self.dec_a(z) * ma, ta * ma)
        # disentangle: shared vs private
        def cos2(a, b):
            c = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)
            return (c ** 2).mean()
        L = L + 0.15 * cos2(zs, torch.cat([zpr, zpa], dim=1))
        # isometric reg: preserve pairwise distances (anti-collapse)
        idx = torch.randperm(z.shape[0], device=DEVICE)[:500]
        Ds = torch.cdist(zs[idx], zs[idx])
        Df = torch.cdist(z[idx], z[idx])
        L = L + 0.1 * F.mse_loss(Ds / Ds.mean().clamp_min(1e-8),
                                 Df / Df.mean().clamp_min(1e-8))
        # spatial (homogeneity-gated)
        L = L + lam_spa * laplacian_loss(z, g["edge_index"])
        # gate entropy: discourage collapse to single modality
        L = L + 0.01 * (-(gw * (gw + 1e-8).log()).sum(dim=1).mean())
        return L

print("S6FrankModel defined")


In [ ]:
# ---------- training + eval ----------
def train_one(data, seed, epochs=60):
    set_seed(seed)
    d = load_dataset(data) if isinstance(data, str) else data
    model = S6FrankModel(d["Xr"].shape[1], d["Xa"].shape[1], d["Xr"].shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    tr, ta = d["Xr"], d["Xa"]
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(d["Xr"], d["Xa"], d["g"])
        loss = model.loss(out, d["Xr"], d["Xa"], tr, ta, d["g"], d["lam_spa"])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        z, zs, zpr, zpa, gw = model(d["Xr"], d["Xa"], d["g"])
    Z = z.cpu().numpy()
    # spherical k-means (Exp0: mouse needs it)
    Zn = normalize(Z, norm="l2")
    k = DECLARED_K[data]
    labs = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Zn)
    # unsupervised metrics
    sil = float(silhouette_score(Zn, labs)) if len(np.unique(labs)) > 1 else -1.0
    # recon balance
    with torch.no_grad():
        rr = F.mse_loss(model.dec_r(z), tr).item()
        ra = F.mse_loss(model.dec_a(z), ta).item()
    recbal = 1.0 - abs(rr - ra) / (rr + ra + 1e-8)
    r2r = float(1 - rr / tr.var().item())
    gw_mean = gw.mean(dim=0).cpu().numpy()
    return Z, labs, dict(silhouette=sil, reconstruction_balance=recbal,
                         r2_rna=r2r, gate_rna=float(gw_mean[0]), gate_aux=float(gw_mean[1]),
                         lam_spa=d["lam_spa"])

# ---------- runner ----------
COMBOS = [("S6-FRANK", {})]
REG = os.path.join(OUT, "registry.jsonl")
done = set()
if os.path.exists(REG):
    for line in open(REG):
        r = json.loads(line)
        done.add((r["combo"], r["dataset"], r["seed"]))

os.makedirs(OUT, exist_ok=True)
for combo, cfg in COMBOS:
    for ds_id in DATASETS:
        for seed in SEEDS:
            if (combo, ds_id, seed) in done:
                print(f"skip {combo} {ds_id} seed{seed}")
                continue
            print(f"\n=== {combo} {ds_id} seed{seed} ===")
            t0 = time.time()
            Z, labs, metrics = train_one(ds_id, seed)
            dd = os.path.join(OUT, combo, ds_id, f"seed{seed}")
            os.makedirs(dd, exist_ok=True)
            np.save(os.path.join(dd, "fused_embedding.npy"), Z)
            np.save(os.path.join(dd, "labels.npy"), labs)
            json.dump(metrics, open(os.path.join(dd, "metrics.json"), "w"), indent=1)
            # quarantined post-hoc ARI
            spec = DATASETS[ds_id]
            p = os.path.join(DATA_ROOT, spec["folder"], spec["gt_file"])
            gt = pd.read_csv(p, index_col=0)[spec["gt_col"]].astype(str).values
            ari = float(adjusted_rand_score(gt, labs))
            json.dump({"ari": ari}, open(os.path.join(dd, "posthoc.json"), "w"))
            print(f"  ARI={ari:.3f} sil={metrics['silhouette']:.3f} "
                  f"gate=[{metrics['gate_rna']:.2f},{metrics['gate_aux']:.2f}] "
                  f"({time.time()-t0:.0f}s)")
            with open(REG, "a") as f:
                f.write(json.dumps({"combo": combo, "dataset": ds_id, "seed": seed}) + "\n")
            git_push([f"runs/s6-frank/{combo}/{ds_id}/seed{seed}", "runs/s6-frank/registry.jsonl"],
                     f"s6-frank: {combo} {ds_id} seed{seed}")
print("\nDone.")


## Notes
- **FRANK** = SAGE + MoE-threshold fusion + shared-private + masked recon + isometric + gated spatial.
- **Firewall:** labels only in posthoc.json. Declared k applied identically, disclosed.
- **Resume:** skips done (combo, dataset, seed) in registry.jsonl.
- **Run:** upload to Kaggle → attach dataset → GITHUB_TOKEN secret → GPU T4 → Run All.
